In [1]:
import os
root = os.getcwd()
while not os.path.exists(os.path.join(root, "Cargo.toml")) and root != "/":
    root = os.path.dirname(root)
os.chdir(root)
import planforge


The goal-count heuristic is the number of propositional goal facts that are not satisfied in a state. It ignores action costs and interactions between goals, but it is simple to compute and useful as a first heuristic example.

In [2]:
task = planforge.Task.from_sas("tests/assets/numeric_sas/example2.sas")
pddl_task = planforge.Task.from_pddl(
    "tests/assets/numeric-pddl-files/fn-counters-small_instances/domain.pddl",
    "tests/assets/numeric-pddl-files/fn-counters-small_instances/problem_2.pddl",
)

for name, current in [("example2.sas", task), ("fn-counters problem_2", pddl_task)]:
    print(name)
    print("  variables:", current.num_variables)
    print("  operators:", current.num_operators)
    print("  goals:", current.num_goals)
    print("  metric:", current.metric)


example2.sas
  variables: 71
  operators: 21
  goals: 14
  metric: True
fn-counters problem_2
  variables: 7
  operators: 4
  goals: 1
  metric: True


The Python version uses the `Task.goals` accessor, which exposes each propositional goal as a `(var, value)` pair.

In [3]:
def goal_count(task, state):
    return sum(1 for (var, val) in task.goals if state.values[var] != val)


In [4]:
s0 = task.initial_state()
print("h(initial):", goal_count(task, s0))
print("successors:")
for op, succ, cost in task.successors(s0):
    print(op.name, goal_count(task, succ))


h(initial): 14
successors:
go_est b0 14
go_north_east b0 14
go_north_west b0 14
go_south b0 14
go_south_east b0 14
go_south_west b0 14
go_west b0 14


Solving with the existing Rust-backed search lets us replay the returned plan and inspect the Python heuristic value along the path.

In [5]:
replay_task = pddl_task
res = replay_task.solve("astar(blind())", max_time=60.0)
print("status:", res.status)
if res.status == "solved":
    state = replay_task.initial_state()
    print("step 0:", goal_count(replay_task, state))
    for step, plan_op in enumerate(res.plan, start=1):
        matching = [succ for op, succ, cost in replay_task.successors(state) if op.name == plan_op.name]
        if len(matching) != 1:
            raise RuntimeError(f"expected one successor for {plan_op.name}, got {len(matching)}")
        state = matching[0]
        print(f"step {step}: {plan_op.name} -> {goal_count(replay_task, state)}")
else:
    print("no solved plan to replay")


status: solved
step 0: 1
step 1: increment c1 -> 0


This mirrors the Rust `GoalCountHeuristic`; the coarse `solve()` runs entirely in Rust.

The same Python function can guide the Rust A* engine directly. The callback receives a state snapshot for each evaluated state.

In [6]:
task = pddl_task

# Guide the Rust A* engine with our Python goal_count heuristic.
guided = task.search_with_heuristic(lambda s: goal_count(task, s), max_time=60.0)
print("guided A*: ", guided.status, "cost", guided.cost, "expanded", guided.nodes_expanded)

blind = task.solve("astar(blind())", max_time=60.0)
print("blind  A*: ", blind.status, "cost", blind.cost, "expanded", blind.nodes_expanded)


guided A*:  solved cost 1.0 expanded 2
blind  A*:  solved cost 1.0 expanded 2


This is the same goal-count idea as the native Rust `GoalCountHeuristic`, but defined in Python and called back per state by the Rust A*.